# Imports

In [1]:
import os
import json
import pandas as pd
from helpers import *

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Wave Question Extraction
Parse raw data sets to extract questions relevant to the research topic. Relevant questions ask respondants about either military aid for Ukraine or economic sanctions on Russia.
Additionally extract fieldwork timelines during which respondants were surveyed and confirm consistency in questions to have a stable DV over time.

In [2]:
collected = []

for file in os.listdir(EB_BASE_PATH):
    collected.append(parse_doc(file))

collected.sort(key=lambda k: k["wave_id"])
df = pd.DataFrame(collected)
df

,eb_n,wave_id,fw_start,fw_end,season,questions,q_ids
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"[QE2.1. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing supply and delivery of military equipment to Ukraine]","[QE2_1, QE2_3]"
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_3]"
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disa

In [3]:
if os.path.exists("data/interim/wave_questions.csv"):
    questions = pd.read_csv("data/interim/wave_questions.csv")
else:
    qs = pd.DataFrame(df["questions"].tolist(), index=df.index)
    qs.columns = ["q1", "q2"]
    qs["q1"] = qs["q1"].apply(lambda v: f"{v.split(' ', 1)[0]} {v.split(':-')[-1]}")
    qs["q2"] = qs["q2"].apply(lambda v: f"{v.split(' ', 1)[0]} {v.split(':-')[-1]}")
    questions = df.join(qs).drop(columns=["questions", "q_ids"])

    with open("data/interim/wave_questions.csv", "w") as f:
        questions.to_csv(f, index=False)

questions

,eb_n,wave_id,fw_start,fw_end,season,q1,q2
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing supply and delivery of military equipment to Ukraine
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.3. Financing the purchase and supply of military equipment to Ukraine
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
8,eb105,105.2,2026-03-12,2026-04-05,Spring 2026,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals (including use of immobilised Russian assets to finance support for Ukraine)",QD2.2. Financing the purchase and supply of military equipment to Ukraine


Looking at the extracted questions from each wave, a change in the wording of the questions only happens twice: *QE2.3* in wave *97.5* and *QD2.1* in wave *105.2*. However, given that the deviations in wording are relatively minor - *97.5-QE2.3* implies the same thing as *98.2-QE2.3* and *105.2-QD2.1* clarifies the scope of the sanctions - a consistent analysis over time can still be made.

# Eurobarometer Outcomes
Per-wave, per-question, per-country response scores extracted from the Eurobarometer datasets

In [4]:
if os.path.exists("data/interim/wave_scores.json"):
    with open("data/interim/wave_scores.json", "r") as f:
        data = json.load(f)
else:
    data = {}

    for row in df[["eb_n", "q_ids"]].itertuples():
        data[row.eb_n] = collect_scores(row.eb_n, row.q_ids)  # type: ignore[arg-type]

    with open("data/interim/wave_scores.json", "w") as f:
        json.dump(data, f, indent=4)

series = pd.Series({
    (wave, q, COUNTRY_CODES[country]): metrics
    for wave, questions in data.items()
    for q, countries in questions.items()
    for country, metrics in countries.items()
})

wdf = pd.DataFrame(series.tolist(), index=series.index)
wdf.index.names = ['wave', 'question', 'country']

pd.reset_option("display.max_rows")
wdf

total  total_agree  total_disagree  totally_agree  \
wave  question country                                                       
eb97  QE2_1    Belgium    1009          819             178            437   
               Bulgaria   1038          477             440            205   
               Czechia    1015          727             249            484   
               Denmark    1037          956              67            762   
               Germany    1507         1229             224            859   
...                        ...          ...             ...            ...   
eb105 QD2_2    Romania    1054          480             538             99   
               Slovenia   1009          368             603             79   
               Slovakia   1003          359             616            125   
               Finland    1009          882             103            502   
               Sweden     1026          953              62            699   

                         tend_to_agree  dont_know  tend_to_disagree  \
wave  question country                                                
eb97  QE2_1    Belgium             382         12               131   
               Bulgaria            272        121               218   
               Czechia             243         39               146   
               Denmark             195         14                52   
               Germany             370         54               149   
...                                ...        ...               ...   
eb105 QD2_2    Romania             381         37               309   
               Slovenia            289         37               360   
               Slovakia            234         29               310   
               Finland             380         24                76   
               Sweden              254         10                41   

                         totally_disagree  
wave  question country                     
eb97  QE2_1    Belgium                 47  
               Bulgaria               222  
               Czechia                103  
               Denmark                 15  
               Germany                 75  
...                                   ...  
eb105 QD2_2    Romania                229  
               Slovenia               243  
               Slovakia               306  
               Finland                 27  
               Sweden                  21  

[486 rows x 8 columns]

# Exposure Coding

Derive the 0-3 exposure tier for each country-window directly from the frozen source register and the codebook observation windows. Criteria A/B/C are hand-coded in the register; D (criterion A corroborated by >=2 independent sources) is computed here as >=2 distinct authors among the sources establishing A. This cell recomputes on every run rather than caching, so the index always reflects the current register (R9). It writes back only the derived columns of the "coding" sheet — manual columns (locators, "ambiguous", bounds, prose notes) and the other sheets are left intact.

In [5]:
register = pd.read_csv("data/interim/source_register.csv")
sources = load_sources(register)

EXPOSURE_XLSX = "data/interim/exposure_coding.xlsx"
coding = pd.read_excel(EXPOSURE_XLSX, sheet_name="coding")

codes = coding.apply(lambda r: code_country_window(
    r["iso2"],
    pd.to_datetime(r["window_start"]).date(),
    pd.to_datetime(r["window_end"]).date(),
    sources), axis=1, result_type="expand")

for col in ["criteria_met", "notes",
            "evidence_1_source", "evidence_2_source", "evidence_3_source"]:
    coding[col] = coding[col].astype("object")

coding["exposure"] = codes["exposure"].astype(int)
coding["criteria_met"] = codes["criteria_met"]

ev = codes["evidence_sources"].str.split(";")
for i in range(3):
    coding[f"evidence_{i+1}_source"] = ev.apply(
        lambda xs: xs[i] if isinstance(xs, list) and len(xs) > i and xs[i] else pd.NA)

# seed note where none has been hand-entered; never overwrite one
blank = coding["notes"].isna() | (coding["notes"].astype(str).str.strip() == "")
coding.loc[blank, "notes"] = codes.loc[blank, "notes"].replace("", pd.NA)

coding["n_sources_A"] = codes["n_sources_A"].astype(int)
coding["n_indep_A"] = codes["n_indep_A"].astype(int)  # the basis for D, stored for defensibility

with pd.ExcelWriter(EXPOSURE_XLSX, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as xl:
    coding.to_excel(xl, sheet_name="coding", index=False)

pd.reset_option("display.max_rows")
coding

,wave_num,eb_n,wave_id,country,iso2,window_start,window_end,exposure,criteria_met,evidence_1_source,evidence_1_locator,evidence_2_source,evidence_2_locator,evidence_3_source,evidence_3_locator,ambiguous,bound_lo,bound_hi,notes,n_sources_A,n_indep_A
0,1,eb97,97.5,Belgium,BE,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,eb97,97.5,Bulgaria,BG,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,eb97,97.5,Czechia,CZ,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,1,eb97,97.5,Denmark,DK,2022-02-24 00:00:00,2022-07-17,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,1,eb97,97.5,Germany,DE,2022-02-24 00:00:00,2022-07-17,3,ABCD,CHK25,NaN,DFR22,NaN,EDL22,NaN,NaN,NaN,NaN,"A:CHK25,DFR22,EDL22,ISD22,MET22,MET24-2 | B:DFR22,EDL22,ISD22,MET22,MET24-2 | C:CHK25,DFR22,EDL22,ISD22,MET22",6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,9,eb105,105.2,Romania,RO,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
239,9,eb105,105.2,Slovenia,SI,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
240,9,eb105,105.2,Slovakia,SK,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
241,9,eb105,105.2,Finland,FI,2025-11-05,2026-04-05,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [6]:
# truncation boundary: waves with no documented exposure anywhere -> drop them (R10)
wave_max = coding.groupby("wave_num")["exposure"].max()
usable = wave_max[wave_max > 0].index.tolist()
truncated = wave_max[wave_max == 0].index.tolist()

# R11 ceiling flag, computed on the truncated panel (all-zero tail masks it otherwise)
trunc = coding[coding["wave_num"].isin(usable)]
var = trunc.groupby("iso2")["exposure"].nunique()
flat = var[var == 1].index.tolist()

print("usable waves:", usable, "| truncate (all-zero):", truncated)
print("no within-country variation (drop out under country FE):", flat)
trunc.pivot(index="iso2", columns="wave_num", values="exposure")

usable waves: [1, 2, 3, 4, 5, 6, 7] | truncate (all-zero): [8, 9]
no within-country variation (drop out under country FE): ['BE', 'DE', 'LU', 'MT']


wave_num,1,2,3,4,5,6,7
iso2,,,,,,,
AT,0,0,0,2,2,0,0
BE,0,0,0,0,0,0,0
BG,0,0,0,0,1,0,0
CY,0,0,0,0,1,0,0
CZ,0,0,0,0,1,0,0
DE,3,3,3,3,3,3,3
DK,0,0,0,0,1,0,0
EE,1,1,0,0,1,0,0
EL,0,0,0,0,1,0,0


Populate source-to-waves crosswalk

In [7]:
waves = (coding[["wave_num", "window_start", "window_end"]]
         .drop_duplicates()
         .sort_values("wave_num"))
waves["window_start"] = pd.to_datetime(waves["window_start"]).dt.date
waves["window_end"] = pd.to_datetime(waves["window_end"]).dt.date

rows = []
for s in sources:
    a, b = s["window"]
    row = {"source_id": s["s_id"]}
    for w in waves.itertuples():
        overlap = bool(a and b and a <= w.window_end and b >= w.window_start)  # type: ignore[arg-type]
        row[f"W{w.wave_num}"] = ";".join(sorted(s["countries"])) if overlap else ""
    rows.append(row)

s2w = pd.DataFrame(rows, columns=["source_id"] + [f"W{n}" for n in waves["wave_num"]])

with pd.ExcelWriter(EXPOSURE_XLSX, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as xl:
    s2w.to_excel(xl, sheet_name="source_to_waves", index=False)

s2w

,source_id,W1,W2,W3,W4,W5,W6,W7,W8,W9
0,EDL22,DE;EE;FR;IT;LT;LV,DE;EE;FR;IT;LT;LV,,,,,,,
1,DFR22,DE;FR;IT;LV;PL,DE;FR;IT;LV;PL,,,,,,,
2,MET22,DE;FR;IT;LV,DE;FR;IT;LV,,,,,,,
3,ISD22,DE;FR;IT,DE;FR;IT,,,,,,,
4,VIG23,,DE;FR,DE;FR,DE;FR,,,,,
5,RST23,,DE;FR;IT,DE;FR;IT,,,,,,
6,MET23,,,DE;FR;PL,DE;FR;PL,,,,,
7,VIG24,,,,AT;DE;ES;FR;PL,AT;DE;ES;FR;PL,,,,
8,AIF24,,,,DE;FR,DE;FR,,,,
9,AIF24-2,,,,,DE;FR;IT;PL,DE;FR;IT;PL,,,


# Controls (Eurostat)
GDP per capita (real), HICP inflation, and Ukrainian displaced-population share, matched to each wave's fieldwork midpoint. Also collects pop2021 for the PSM moderator.

In [8]:
codes = list(COUNTRY_CODES.keys())

if os.path.exists("data/interim/controls.csv") and os.path.exists("data/interim/pop2021.csv"):
    controls = pd.read_csv("data/interim/controls.csv")
    pop2021  = pd.read_csv("data/interim/pop2021.csv").set_index("iso2")["population"].to_dict()
else:
    # wave reference: fieldwork midpoint -> (year, YYYY-MM)
    wr = questions[["eb_n","fw_start","fw_end"]].copy()
    wr[["fw_start","fw_end"]] = wr[["fw_start","fw_end"]].apply(pd.to_datetime)
    wr["mid"]  = wr.fw_start + (wr.fw_end - wr.fw_start)/2
    wr["year"] = wr.mid.dt.year
    wr["ym"]   = wr.mid.dt.strftime("%Y-%m")
    wr = wr.merge(coding[["eb_n","wave_num"]].drop_duplicates(), on="eb_n")

    # GDP per capita, real (chain-linked volume, EUR per head), annual
    g = eurostat_long("nama_10_pc", geos=codes, dims={"na_item": "B1GQ"}, since="2021")
    clv = sorted(u for u in g["unit"].unique() if u.startswith("CLV") and u.endswith("_EUR_HAB"))
    g = g[g["unit"] == clv[-1]]
    gdp = g.assign(year=lambda d: d.period.astype(int)).rename(columns={"value":"gdp_pc"})[
          ["geo","year","gdp_pc"]]

    # HICP, monthly annual rate of change, all-items
    h = eurostat_long("prc_hicp_manr", geos=codes, dims={"coicop": "CP00"}, since="2022-01")
    hicp = h.rename(columns={"period":"ym","value":"hicp"})[["geo","ym","hicp"]]

    # Ukrainian temporary-protection beneficiaries (monthly stock)
    b = eurostat_long("migr_asytpsm", geos=codes,
                      dims={"citizen":"UA","sex":"T","age":"TOTAL"}, since="2022-01")
    benef = b.rename(columns={"period":"ym","value":"benef"})[["geo","ym","benef"]]

    # population (annual, 1 Jan)
    p = eurostat_long("demo_pjan", geos=codes, dims={"sex":"T","age":"TOTAL"}, since="2021")
    pop = p.assign(year=lambda d: d.period.astype(int)).rename(columns={"value":"population"})[
          ["geo","year","population"]]

    # assemble per (country, wave)
    base = pd.DataFrame({"geo": codes}).merge(wr[["wave_num","year","ym"]], how="cross")
    controls = (base
        .merge(gdp,   on=["geo","year"], how="left")
        .merge(hicp,  on=["geo","ym"],   how="left")
        .merge(benef, on=["geo","ym"],   how="left")
        .merge(pop,   on=["geo","year"], how="left"))
    controls["displaced_share"] = 100 * controls["benef"].fillna(0) / controls["population"]
    controls = controls.rename(columns={"geo":"iso2"})[
        ["iso2","wave_num","gdp_pc","hicp","displaced_share"]]

    pop2021 = pop[pop.year==2021].set_index("geo")["population"].to_dict()

    controls.to_csv("data/interim/controls.csv", index=False)
    pd.Series(pop2021, name="population").rename_axis("iso2").to_csv("data/interim/pop2021.csv")

controls

,iso2,wave_num,gdp_pc,hicp,displaced_share
0,BE,1,43890.0,10.4,0.439290
1,BE,2,44190.0,7.4,0.520319
2,BE,3,44190.0,1.6,0.563111
3,BE,4,44190.0,-0.8,0.613866
4,BE,5,44410.0,4.9,0.651048
...,...,...,...,...,...
238,SE,5,48770.0,2.4,0.370367
239,SE,6,48770.0,1.6,0.425760
240,SE,7,49310.0,2.1,0.340442
241,SE,8,49310.0,3.1,0.452459


# PSM Moderator
Per-capita public PSM revenues (EUR, 2021). Totals transcribed from Figure 4 of "Public financing of news media in the EU" (source: EAO database); population is pulled from Eurostat (demo_pjan 2021) in the Controls cell above. The assert gate reproduces the report's anchors (DE 103.0, RO 7.6) before saving.

In [9]:
if os.path.exists("data/interim/psm.csv"):
    psm = pd.read_csv("data/interim/psm.csv")
else:
    total_meur = {  # Figure 4, EUR million, 2021 (Malta 2019) — EL is Greece
        "DE":8565.5,"FR":3469.3,"ES":2125.9,"IT":1819.8,"SE":836.3,"NL":697.2,"AT":645.2,
        "BE":590.8,"PL":525.1,"FI":494.1,"DK":462.9,"CZ":311.4,"HU":282.2,"IE":233.0,
        "EL":188.3,"PT":181.5,"HR":154.4,"RO":146.5,"SK":123.8,"SI":102.2,"BG":81.4,
        "LT":52.3,"EE":41.1,"LV":35.4,"CY":34.0,"LU":6.9,"MT":4.2}

    psm = pd.DataFrame([{"iso2": c, "psm": round(total_meur[c]*1e6/pop2021[c], 1)}
                        for c in total_meur])
    v = psm.set_index("iso2")["psm"]
    assert abs(v["DE"]-103.0) < 0.6 and abs(v["RO"]-7.6) < 0.3, "PSM anchors failed"
    psm.to_csv("data/interim/psm.csv", index=False)

psm.sort_values("psm", ascending=False)

,iso2,psm
0,DE,103.0
9,FI,89.3
4,SE,80.6
10,DK,79.3
6,AT,72.2
1,FR,51.2
7,BE,51.1
19,SI,48.5
13,IE,46.0
2,ES,44.8


# Panel Assembly
Collapse everything to one country-wave table for the models: exposure (usable waves only), the two DVs reduced to % agree, and the merged controls and PSM moderator.

In [10]:
# exposure, truncated to waves with documented coverage (W1-W7)
usable = coding.groupby("wave_num")["exposure"].max().pipe(lambda s: s[s > 0].index)
exposure = coding[coding["wave_num"].isin(usable)][["iso2", "wave_num", "eb_n", "exposure"]]

# DV: classify each wave's two items by keyword, then take total_agree / total
def qid_of(text):
    return text.split()[0].rstrip(".").replace(".", "_")

roles = {}
for r in questions.itertuples():
    m = {}
    for txt in (r.q1, r.q2):
        role = ("sanctions" if "sanction" in txt.lower()  # type: ignore
                else "aid" if "military" in txt.lower() else None)  # type: ignore
        if role:
            m[role] = qid_of(txt)
    roles[r.eb_n] = m

dv = pd.DataFrame([
    {"eb_n": e, "iso2": iso2,
     "support_aid": 100 * data[e][q["aid"]][iso2]["total_agree"] / data[e][q["aid"]][iso2]["total"],
     "support_sanctions": 100 * data[e][q["sanctions"]][iso2]["total_agree"] / data[e][q["sanctions"]][iso2]["total"]}
    for e, q in roles.items() for iso2 in data[e][q["aid"]]
])

# merge exposure + DV + controls + PSM, all keyed on iso2 / wave_num
panel = (exposure
    .merge(dv, on=["eb_n", "iso2"], how="left", validate="one_to_one")
    .merge(controls, on=["iso2", "wave_num"], how="left", validate="one_to_one")
    .merge(psm, on="iso2", how="left"))

# standardize the moderator over the 27 unique countries (cross-country SD), not panel rows
u = panel.drop_duplicates("iso2")["psm"]
panel["psm_z"] = (panel["psm"] - u.mean()) / u.std()

assert not panel.duplicated(["iso2", "wave_num"]).any()
panel.to_csv("data/interim/panel.csv", index=False)
panel

,iso2,wave_num,eb_n,exposure,support_aid,support_sanctions,gdp_pc,hicp,displaced_share,psm,psm_z
0,BE,1,eb97,0,72.547076,81.169475,43890.0,10.4,0.439290,51.1,0.462371
1,BG,1,eb97,0,34.971098,45.953757,10740.0,14.9,1.932515,12.5,-1.007682
2,CZ,1,eb97,0,56.256158,71.625616,21910.0,17.3,3.679812,29.7,-0.352632
3,DK,1,eb97,0,90.935391,92.189007,56530.0,9.6,0.445396,79.3,1.536348
4,DE,1,eb97,3,70.670206,81.552754,44230.0,8.5,0.000000,103.0,2.438945
...,...,...,...,...,...,...,...,...,...,...,...
184,RO,7,eb103,0,51.010587,70.452358,13190.0,4.9,0.957352,7.6,-1.194295
185,SI,7,eb103,0,41.641939,54.104847,25680.0,2.3,0.482437,48.5,0.363352
186,SK,7,eb103,0,45.373134,61.393035,19270.0,3.9,2.432996,22.7,-0.619222
187,FI,7,eb103,0,90.268123,89.473684,43450.0,1.9,1.248321,89.3,1.917190


# Models
The core estimates. For each outcome — support for military aid and for sanctions — four two-way fixed-effects specifications build up from the bare exposure term (m1) to the full model with controls and the PSM interaction (m4). Country and wave fixed effects absorb all stable national differences and common wave shocks, so every coefficient is identified within-country, net of common trends. Standard errors are clustered by country; because 27 clusters is few enough that clustered SEs over-reject, the block below the tables reports wild cluster restricted bootstrap p-values for the exposure term, which stay valid with a small number of clusters.

In [11]:
import pyfixest as pf
vcov, ctrls = {"CRV1": "iso2"}, "gdp_pc + hicp + displaced_share"

fits = {}
tables = {}

for dv in ["support_aid", "support_sanctions"]:
    fits[dv] = {
        "m1 base": pf.feols(f"{dv} ~ exposure | iso2 + wave_num", panel, vcov=vcov),  # type: ignore
        "m2 +controls": pf.feols(f"{dv} ~ exposure + {ctrls} | iso2 + wave_num", panel, vcov=vcov),  # type: ignore
        "m3 +psm": pf.feols(f"{dv} ~ exposure + exposure:psm_z | iso2 + wave_num", panel, vcov=vcov),  # type: ignore
        "m4 full": pf.feols(f"{dv} ~ exposure + exposure:psm_z + {ctrls} | iso2 + wave_num", panel, vcov=vcov),  # type: ignore
    }

print("\nWild cluster restricted bootstrap p-values (H0: exposure = 0):")

for dv in ["support_aid", "support_sanctions"]:
    for name, c in [("m1", ()), ("m2", ("gdp_pc", "hicp", "displaced_share"))]:
        b, p = wcr_wildboot(panel, dv, controls=c)
        print(f"  {dv:18s} {name}:  b={b:+.3f}   p_WCR={p:.3f}")


Wild cluster restricted bootstrap p-values (H0: exposure = 0):
  support_aid        m1:  b=+0.292   p_WCR=0.706
  support_aid        m2:  b=+0.476   p_WCR=0.569
  support_sanctions  m1:  b=-0.015   p_WCR=0.971
  support_sanctions  m2:  b=+0.082   p_WCR=0.848


In [12]:
pf.etable(fits["support_aid"].values())

GT(_tbl_data=  __index_level_0__ __index_level_1__                   0  \
0              coef          exposure  0.292 <br> (0.715)   
1              coef            gdp_pc                       
2              coef              hicp                       
3              coef   displaced_share                       
4              coef  exposure × psm_z                       
5                fe          wave_num                   x   
6                fe             iso2                    x   
7             stats      Observations                 189   
8             stats                R²               0.959   

                     1                    2                    3  
0   0.476 <br> (0.755)   0.254 <br> (0.673)   0.439 <br> (0.706)  
1  -0.001 <br> (0.001)                       -0.001 <br> (0.001)  
2  -0.140 <br> (0.144)                       -0.144 <br> (0.147)  
3  -1.613 <br> (1.375)                       -1.635 <br> (1.368)  
4                       -0.738 <br> (0.685)  -0.757 <br> (0.683)  
5                    x                    x                    x  
6                    x                    x                    x  
7                  189                  189                  189  
8                 0.96                0.959                0.961  , _body=<great_tables._gt_data.Body object at 0x11c651010>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11c5f82d0>, _spanners=Spanners([SpannerInfo(spanner_id='support_aid', spanner_level=1, spanner_label='support_aid', spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11c651400>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11c5f9810>, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell: Coefficient   (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11c651550>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), t

## Reading the model output

The tables above report the four fixed-effects specifications (m1–m4) for each outcome; the block beneath them reports wild cluster bootstrap p-values. Both DVs ultimately tell the same story.

### Regression tables

Each column is one specification; each cell is **coefficient (standard error)**. A coefficient is distinguishable from zero only if it exceeds roughly twice its standard error, and is noise otherwise. The `iso2` and `wave_num` rows marked `x` are absorbed fixed effects, so every estimate is *within-country, net of common wave shocks*.

- **`exposure`** tests H1/H2 — the change in percentage-point support per one-tier increase in documented exposure. Across all four columns it is a fraction of a point, smaller than its own standard error, and for aid it is *positive*, the opposite of the hypothesised erosion. It barely moves as controls and the interaction are added, marking it as near zero rather than masked.
- **`exposure × psm_z`** tests H3 (does PSM funding moderate exposure). Non-significant, and its negative sign would imply *more* erosion in better-funded systems — against the buffering hypothesis, not for it.
- **Controls** (`gdp_pc`, `hicp`, `displaced_share`) appear only where included; none is significant, and they are nuisance parameters, not the object of interest.
- **N = 189** (27 countries × 7 usable waves) confirms the balanced, truncated panel.
- **R² ≈ 0.96 is not explanatory power.** The 34 fixed-effect dummies mechanically absorb almost all outcome variation; the number reflects that countries differ and support trended in common, and says nothing about exposure. Ignore it for interpretation.
- The significance footnote (`* p<0.05 …`) makes the finding visible: **no star appears on any exposure or interaction coefficient.**

### Bootstrap p-values

Clustered standard errors assume many clusters; with 27 they run too small and over-reject.
The wild cluster restricted bootstrap rebuilds each p-value under an imposed null by
sign-flipping whole countries 9,999 times, giving inference valid for few clusters. The
values (0.57–0.97) come out *slightly larger* than the analytic p-values — the small-cluster
correction admitting honest uncertainty. The null survives the stricter test; the bootstrap
confirms it rather than rescuing significance.

### Summary

> Across both outcomes and every specification, documented Doppelgänger exposure has **no
> statistically detectable association** with support for aid or sanctions — estimates are a
> fraction of a point, sometimes the wrong sign, and robust to small-cluster inference. H1,
> H2, and H3 are unsupported.

This is *no effect detected*, not *no effect proven*. Power is limited by design: Germany
sits at the exposure ceiling (no within-country variation, so it contributes nothing under
country fixed effects), and France/Italy/Poland move only on the single, reporting-lag-
affected final wave. The exposure measure also captures documented campaign *activity*, not
consumed *reach*. The null is credible precisely because these limits are named rather than
hidden.

# Robustness
The null in the core models should survive reasonable perturbations of the design. Four checks:

1. **Drop W7.** The final wave carries a reporting-lag risk – late forensic attribution can post-date fieldwork, so a W7 exposure spike may be artifactual. Re-estimate on W1–W6.
2. **Renormalized DV.** The primary outcome keeps *don't-knows* in the denominator (`agree / total`). This re-runs on `agree / (agree + disagree)`, testing whether the null depends on the normalization choice.
3. **Don't-know outcome.** If exposure demobilizes opinion rather than shifting it, the effect would surface as a rise in the *don't-know* share, not in support. This regresses DK share directly on exposure – the one channel where a signal could hide.
4. **Country-specific linear trends.** Adds a separate linear time trend per country, absorbing divergent national trajectories. Demanding on 7 waves and reported analytically since the bootstrap is over-parameterized here.

In [13]:
# alternative outcomes on a copy of the panel: renormalized (DK dropped) and DK share
def _dv(e, q, iso2, kind):
    m = data[e][q[kind]][iso2]
    return m["total_agree"], m["total_disagree"], m["dont_know"], m["total"]

_aug = pd.DataFrame([
    {"eb_n": e, "iso2": iso2,
     "support_aid_rn": 100 * _dv(e, q, iso2, "aid")[0] / sum(_dv(e, q, iso2, "aid")[:2]),
     "support_sanctions_rn": 100 * _dv(e, q, iso2, "sanctions")[0] / sum(_dv(e, q, iso2, "sanctions")[:2]),
     "dk_aid": 100 * _dv(e, q, iso2, "aid")[2] / _dv(e, q, iso2, "aid")[3],
     "dk_sanctions": 100 * _dv(e, q, iso2, "sanctions")[2] / _dv(e, q, iso2, "sanctions")[3]}
    for e, q in roles.items() for iso2 in data[e][q["aid"]]
])

rob = panel.merge(_aug, on=["eb_n", "iso2"], how="left", validate="one_to_one")

In [14]:
import warnings

ctrls_t = ("gdp_pc", "hicp", "displaced_share")

rows = []
for label, frame, dvs in [
    ("R1 drop W7", rob[rob.wave_num < 7], ["support_aid", "support_sanctions"]),
    ("R2 renormalized DV", rob, ["support_aid_rn", "support_sanctions_rn"]),
    ("R3 don't-know outcome", rob, ["dk_aid", "dk_sanctions"]),
]:
    for dv in dvs:
        b, p = wcr_wildboot(frame, dv, controls=ctrls_t)
        rows.append([label, dv, "WCR bootstrap", round(b, 3), round(p, 3), len(frame)])

# R4 country-specific linear trends – analytic CRV1 (bootstrap over-parameterized with 27 trends).
# Slovakia's trend is collinear with the FE block and is dropped; expected, not an error.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for dv in ["support_aid", "support_sanctions"]:
        m = pf.feols(f"{dv} ~ exposure + gdp_pc + hicp + displaced_share + C(iso2):wave_num"
                     f" | iso2 + wave_num", rob, vcov={"CRV1": "iso2"})  # type: ignore[arg-type]
        rows.append(["R4 country trends", dv, "analytic CRV1",
                     round(m.coef()["exposure"], 3), round(m.pvalue()["exposure"], 3), len(rob)])

rob_tbl = pd.DataFrame(rows, columns=["check", "outcome", "inference", "beta", "p", "N"])
rob_tbl

,check,outcome,inference,beta,p,N
0,R1 drop W7,support_aid,WCR bootstrap,-0.051,0.957,162
1,R1 drop W7,support_sanctions,WCR bootstrap,-0.071,0.913,162
2,R2 renormalized DV,support_aid_rn,WCR bootstrap,0.598,0.488,189
3,R2 renormalized DV,support_sanctions_rn,WCR bootstrap,0.094,0.829,189
4,R3 don't-know outcome,dk_aid,WCR bootstrap,0.230,0.207,189
5,R3 don't-know outcome,dk_sanctions,WCR bootstrap,0.027,0.893,189
6,R4 country trends,support_aid,analytic CRV1,0.483,0.513,189
7,R4 country trends,support_sanctions,analytic CRV1,0.184,0.660,189


**Reading the table.** Each row is one robustness specification; the exposure coefficient and its p-value are the objects of interest.

- **`check`** — which perturbation of the design: `R1` restricts to W1–W6, `R2`/`R3` swap the outcome, `R4` adds per-country linear trends.
- **`outcome`** — the dependent variable for that row:
  - `support_aid` / `support_sanctions` — baseline outcomes, % agreeing with don't-knows kept in the denominator (`agree / total`).
  - `_rn` suffix — *renormalized*: don't-knows dropped, `agree / (agree + disagree)`.
  - `dk_` prefix — the *don't-know share*, `dont_know / total`, used as the demobilization outcome.
- **`inference`** — how the p-value is computed: `WCR bootstrap` is the wild cluster restricted bootstrap (9,999 sign-flips, valid with few clusters); `analytic CRV1` is the standard country-clustered p-value, used for R4 where the bootstrap is over-parameterized.
- **`beta`** — the estimated change in the outcome (in percentage points) per one-tier increase in documented exposure.
- **`p`** — two-sided p-value for the null that exposure has no effect (`H0: β = 0`), under the stated inference method.
- **`N`** — number of country-wave observations in that specification (162 when W7 is dropped, 189 otherwise).

Each robustness check re-runs the m2 specification (exposure plus controls, with country and wave fixed effects) on a modified panel or outcome, and reports the wild cluster bootstrap p-value for the exposure term. R1 restricts the sample to W1–W6; R2 and R3 swap the outcome for the renormalized support share and the don't-know share respectively; R4 adds per-country linear trends and is reported analytically, since bootstrapping 27 trends is over-parameterized. A stable, non-significant exposure coefficient across all four is what confirmation of the null looks like.

All four checks confirm the null. Dropping W7 moves the aid coefficient from +0.48 to −0.05 (p=0.96) – thus the small positive baseline estimate was an artifact of the reporting-lag wave, not evidence of an effect. The renormalized DV and country-trend specifications leave the null intact. The one estimate that leans toward the hypothesised direction is the **don't-know share for aid** (+0.23, p=0.21): exposure is associated with marginally more opinion withholding, consistent with a weak demobilization channel, but it is **not statistically significant** and does not disturb the headline result. Reported honestly as suggestive, not detected.

> The R4 spec drops one country trend (Slovakia) as collinear with the fixed-effect block. This is mechanical – 27 country trends plus country and wave fixed effects on 7 waves leave one redundant – and does not affect the exposure estimate.